# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [ ]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import yaml
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut
from geopy.extra.rate_limiter import RateLimiter

In [ ]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [ ]:
# Nominatim's usage policy caps requests at 1/second, so space requests out
# and retry on the 429s that a bare geocode() loop hits.
geocoder = Nominatim(user_agent="academicpages.github.io", timeout=10)
geocode = RateLimiter(geocoder.geocode, min_delay_seconds=1.5, max_retries=5, error_wait_seconds=5.0)
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [ ]:
# Perform geolocation
for file in sorted(g):
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    year = data['date'].year
    description = f"{title}<br />{venue}; {location} ({year})"

    # Geocode the location and report the status
    try:
        location_dict[description] = geocode(location)
        print(description, location_dict[description])
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

In [ ]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

# Institutions worked at get a yellow pin instead of the default blue one.
INSTITUTION_KEYWORDS = [
    "Université de Bordeaux",
    "University of Florida",
    "University of Illinois at Urbana-Champaign",
    "Swansea University",
    "Manchester Institute of Biotechnology",
    "Manchester Metropolitan University",
]

# getorg's own map.html template doesn't know about institution highlighting,
# doesn't style the Leaflet attribution control to match this site's font,
# and its instructional <span> doesn't match this site's fonts either.
# Rewrite the generated map.html with our own fixed template instead of
# hand-patching getorg's output, so every regeneration (including CI)
# produces the same result regardless of what getorg's template looks like.
MAP_HTML = """
    <!DOCTYPE html>
    <html>
    <head>
    \t<meta charset="utf-8" />
    \t<title>Leaflet debug page</title>

    \t<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.0.0-beta.2/leaflet.css" />
    \t<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.0.0-beta.2/leaflet.js"></script>
    \t<meta name="viewport" content="width=device-width, initial-scale=1.0">
    \t<link rel="stylesheet" href="leaflet_dist/screen.css" />

    \t<link rel="stylesheet" href="leaflet_dist/MarkerCluster.css" />
    \t<link rel="stylesheet" href="leaflet_dist/MarkerCluster.Default.css" />
    \t<script src="leaflet_dist/leaflet.markercluster-src.js"></script>
    \t<script src="org-locations.js"></script>
    \t<style>
    \t\tbody, .leaflet-container {
    \t\t\tfont-family: -apple-system, BlinkMacSystemFont, 'Inter', 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
    \t\t}
    \t</style>

    </head>
    <body>

    \t<div id="map"></div>
    \t<script type="text/javascript">
    \t\tvar tiles = L.tileLayer('http://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}', {
              maxZoom: 18,
              attribution: 'Tiles &copy; Esri &mdash; Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China (Hong Kong), Esri (Thailand), TomTom, 2012'
                    }),
    \t\t\tlatlng = L.latLng(30, 10);
    \t\tvar map = L.map('map', {center: latlng, zoom: 0.7, layers: [tiles]});
    \t\tvar markers = L.markerClusterGroup({
    \t\t\tshowCoverageOnHover: false,
    \t\t\tmaxClusterRadius: 80
    \t\t\t});

    \t\t// Institutions worked at get a yellow pin instead of the default blue one.
    \t\tvar institutionKeywords = %(institution_keywords)s;
    \t\tvar institutionIcon = L.divIcon({
    \t\t\tclassName: 'institution-marker',
    \t\t\thtml: '<svg width="25" height="41" viewBox="0 0 25 41" xmlns="http://www.w3.org/2000/svg">' +
    \t\t\t\t'<path d="M12.5 0C5.6 0 0 5.6 0 12.5c0 9.4 12.5 28.5 12.5 28.5S25 21.9 25 12.5C25 5.6 19.4 0 12.5 0z" fill="#FFC107" stroke="#8a6d00" stroke-width="1"/>' +
    \t\t\t\t'<circle cx="12.5" cy="12.5" r="5" fill="#ffffff"/></svg>',
    \t\t\ticonSize: [25, 41],
    \t\t\ticonAnchor: [12, 41],
    \t\t\tpopupAnchor: [1, -34]
    \t\t});

    \t\tfor (var i = 0; i < addressPoints.length; i++) {
    \t\t\tvar a = addressPoints[i];
    \t\t\tvar title = a[0];
    \t\t\tvar isInstitution = institutionKeywords.some(function (keyword) {
    \t\t\t\treturn title.indexOf(keyword) !== -1;
    \t\t\t});
    \t\t\tvar markerOptions = { title: title };
    \t\t\tif (isInstitution) {
    \t\t\t\tmarkerOptions.icon = institutionIcon;
    \t\t\t}
    \t\t\tvar marker = L.marker(new L.LatLng(a[1], a[2]), markerOptions);
    \t\t\tmarker.bindPopup(title);
    \t\t\tmarkers.addLayer(marker);
    \t\t}
    \t\tmap.addLayer(markers);
    \t\tmap.zoomIn();
    \t</script>
    </body>
    </html>
    """

keywords_js = "[\n" + ",\n".join(f"\t\t\t'{kw}'" for kw in INSTITUTION_KEYWORDS) + "\n\t\t]"
with open("talkmap/map.html", "w") as f:
    f.write(MAP_HTML % {"institution_keywords": keywords_js})

In [ ]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

# Institutions worked at get a yellow pin instead of the default blue one.
INSTITUTION_KEYWORDS = [
    "University of Illinois at Urbana-Champaign",
    "Manchester Institute of Biotechnology",
    "Manchester Metropolitan University",
]

# getorg's own map.html template doesn't know about institution highlighting,
# and its instructional <span> doesn't match this site's fonts. Rewrite the
# generated map.html with our own fixed template instead of hand-patching
# getorg's output, so every regeneration (including CI) produces the same
# result regardless of what getorg's template looks like.
MAP_HTML = """
    <!DOCTYPE html>
    <html>
    <head>
    \t<title>Leaflet debug page</title>

    \t<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.0.0-beta.2/leaflet.css" />
    \t<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.0.0-beta.2/leaflet.js"></script>
    \t<meta name="viewport" content="width=device-width, initial-scale=1.0">
    \t<link rel="stylesheet" href="leaflet_dist/screen.css" />

    \t<link rel="stylesheet" href="leaflet_dist/MarkerCluster.css" />
    \t<link rel="stylesheet" href="leaflet_dist/MarkerCluster.Default.css" />
    \t<script src="leaflet_dist/leaflet.markercluster-src.js"></script>
    \t<script src="org-locations.js"></script>

    </head>
    <body>

    \t<div id="map"></div>
    \t<script type="text/javascript">
    \t\tvar tiles = L.tileLayer('http://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}', {
              maxZoom: 18,
              attribution: 'Tiles &copy; Esri &mdash; Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China (Hong Kong), Esri (Thailand), TomTom, 2012'
                    }),
    \t\t\tlatlng = L.latLng(30, 10);
    \t\tvar map = L.map('map', {center: latlng, zoom: 0.7, layers: [tiles]});
    \t\tvar markers = L.markerClusterGroup({
    \t\t\tshowCoverageOnHover: false,
    \t\t\tmaxClusterRadius: 80
    \t\t\t});

    \t\t// Institutions worked at get a yellow pin instead of the default blue one.
    \t\tvar institutionKeywords = %(institution_keywords)s;
    \t\tvar institutionIcon = L.divIcon({
    \t\t\tclassName: 'institution-marker',
    \t\t\thtml: '<svg width="25" height="41" viewBox="0 0 25 41" xmlns="http://www.w3.org/2000/svg">' +
    \t\t\t\t'<path d="M12.5 0C5.6 0 0 5.6 0 12.5c0 9.4 12.5 28.5 12.5 28.5S25 21.9 25 12.5C25 5.6 19.4 0 12.5 0z" fill="#FFC107" stroke="#8a6d00" stroke-width="1"/>' +
    \t\t\t\t'<circle cx="12.5" cy="12.5" r="5" fill="#ffffff"/></svg>',
    \t\t\ticonSize: [25, 41],
    \t\t\ticonAnchor: [12, 41],
    \t\t\tpopupAnchor: [1, -34]
    \t\t});

    \t\tfor (var i = 0; i < addressPoints.length; i++) {
    \t\t\tvar a = addressPoints[i];
    \t\t\tvar title = a[0];
    \t\t\tvar isInstitution = institutionKeywords.some(function (keyword) {
    \t\t\t\treturn title.indexOf(keyword) !== -1;
    \t\t\t});
    \t\t\tvar markerOptions = { title: title };
    \t\t\tif (isInstitution) {
    \t\t\t\tmarkerOptions.icon = institutionIcon;
    \t\t\t}
    \t\t\tvar marker = L.marker(new L.LatLng(a[1], a[2]), markerOptions);
    \t\t\tmarker.bindPopup(title);
    \t\t\tmarkers.addLayer(marker);
    \t\t}
    \t\tmap.addLayer(markers);
    \t\tmap.zoomIn();
    \t</script>
    </body>
    </html>
    """

keywords_js = "[\n" + ",\n".join(f"\t\t\t'{kw}'" for kw in INSTITUTION_KEYWORDS) + "\n\t\t]"
with open("talkmap/map.html", "w") as f:
    f.write(MAP_HTML % {"institution_keywords": keywords_js})